<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 🏗️ EarthDaily Agriculture - Entity Management

**Operations:** Create | Modify | Delete | Export entities (Farm, Field, Seasonfield)

**Hierarchy:** `Grower → Farm → Field → Seasonfield`

**Input File Columns (CSV/Excel/SHP):**
- `Grower` → Grower ID (alphanumeric, use Lookup to find it)
- `Farm` → Farm name (reused if repeated)
- `Seasonfield` → Field/Seasonfield name
- `Sowing` → Sowing date (YYYY-MM-DD)
- `Crop` → Crop code (e.g., CORN, WHEAT - use Lookup to find codes)
- `Geometry` → WKT geometry (auto-extracted for SHP files)

---

## ✅ Step 1: Initialization

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
from earthdaily.agriculture.services.user_management import UserManager
from earthdaily.agriculture.services.entity_management import EntityManager
import os

# Initialize workflow
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

# Initialize UserManager (for Grower lookup)
user_mgr = UserManager(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
    workflow_ref=manager
)

# Initialize EntityManager (with UserManager for Grower resolution)
entity_mgr = EntityManager(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
    workflow_ref=manager,
    user_manager=user_mgr
)

print("✅ EntityManager ready")

## 🔍 Step 2: Lookup IDs

Use this section to find IDs needed for your input file:
- **Growers**: Find Grower IDs by login
- **Farms**: Find existing Farm IDs
- **Fields**: Find existing Field IDs
- **Seasonfields**: Find existing Seasonfield IDs
- **Crops**: Find available Crop codes

In [ ]:
# ============ CONFIGURATION ============
LOOKUP_TYPE = "growers"  # Options: "growers", "farms", "fields", "seasonfields", "crops"

# Optional filters (set to None to skip)
GROWER_ID_FILTER = None      # Filter farms/fields by grower ID
FARM_ID_FILTER = None        # Filter fields by farm ID
FIELD_ID_FILTER = None       # Filter seasonfields by field ID
CROP_CODE_FILTER = None      # Filter seasonfields by crop code
# ========================================

if LOOKUP_TYPE == "growers":
    print("👤 Fetching Growers...")
    df_lookup = entity_mgr.get_growers()
    
elif LOOKUP_TYPE == "farms":
    print("🏠 Fetching Farms...")
    df_lookup = entity_mgr.get_farms(grower_id=GROWER_ID_FILTER)
    
elif LOOKUP_TYPE == "fields":
    print("🌾 Fetching Fields...")
    df_lookup = entity_mgr.get_fields(farm_id=FARM_ID_FILTER)
    
elif LOOKUP_TYPE == "seasonfields":
    print("📅 Fetching Seasonfields...")
    df_lookup = entity_mgr.get_seasonfields(field_id=FIELD_ID_FILTER, crop_code=CROP_CODE_FILTER)
    
elif LOOKUP_TYPE == "crops":
    print("🌽 Fetching Crops...")
    df_lookup = entity_mgr.get_crops()
    
else:
    raise ValueError(f"Unknown LOOKUP_TYPE: {LOOKUP_TYPE}")

print(f"\n📊 Retrieved {len(df_lookup)} {LOOKUP_TYPE}")
display(df_lookup.head(20))

## 📂 Step 3: Build / Load Input Data

Either build the DataFrame inline (default — no external dependency) or load from a file.

### Required columns

| Column        | Type   | Description                                               | Example                                |
|---------------|--------|-----------------------------------------------------------|----------------------------------------|
| `Grower`      | str    | Grower ID (alphanumeric — find via Step 2 lookup)         | `v5vvr6r`                              |
| `Farm`        | str    | Farm name (reused if repeated → single Farm created)      | `Fazenda Boa Vista`                    |
| `Seasonfield` | str    | Field / Seasonfield name                                  | `Talhao_01`                            |
| `Sowing`      | str    | Sowing date `YYYY-MM-DD`                                  | `2025-10-15`                           |
| `Crop`        | str    | Crop code (find available codes via Step 2 `crops` lookup)| `SOYBEANS`, `CORN`, `WHEAT`            |
| `Geometry`    | str    | WKT polygon (auto-extracted for `.shp`)                   | `POLYGON((-55.715 -12.555, ...))`      |

### File formats supported by `load_input_file()`
- **CSV** — auto-detects separator (`,`, `;`, `\t`, `|`)
- **Excel** — `.xlsx`, `.xls`
- **Shapefile** — `.shp` (geometry auto-extracted to WKT)

In [ ]:
import pandas as pd

# ============ CONFIGURATION ============
USE_TEST_DATA = True                 # True = built-in test dataset, False = load from file
GROWER_ID = "v5vvr6r"                # Replace with your Grower ID (from Step 2 lookup)

# File-loading config (used when USE_TEST_DATA = False)
INPUT_FILE = "entities_input.csv"    # CSV / Excel / SHP
INPUT_FOLDER = "inputs"
# ========================================

if USE_TEST_DATA:
    # 3 fields, 2 farms — second farm has 2 fields to demo farm-cache reuse.
    # WKT polygons are ~25 ha squares around Sorriso, Mato Grosso, Brazil.
    df_input = pd.DataFrame([
        {
            "Grower": GROWER_ID,
            "Farm": "Fazenda Boa Vista",
            "Seasonfield": "Talhao_01",
            "Sowing": "2025-10-15",
            "Crop": "SOYBEANS",
            "Geometry": "POLYGON((-55.715 -12.555, -55.710 -12.555, -55.710 -12.550, -55.715 -12.550, -55.715 -12.555))",
        },
        {
            "Grower": GROWER_ID,
            "Farm": "Fazenda Sao Joao",
            "Seasonfield": "Talhao_A",
            "Sowing": "2025-10-20",
            "Crop": "SOYBEANS",
            "Geometry": "POLYGON((-55.700 -12.560, -55.695 -12.560, -55.695 -12.555, -55.700 -12.555, -55.700 -12.560))",
        },
        {
            "Grower": GROWER_ID,
            "Farm": "Fazenda Sao Joao",
            "Seasonfield": "Talhao_B",
            "Sowing": "2026-02-10",
            "Crop": "CORN",
            "Geometry": "POLYGON((-55.730 -12.570, -55.725 -12.570, -55.725 -12.565, -55.730 -12.565, -55.730 -12.570))",
        },
    ])
    print(f"🧪 Using built-in test dataset: {len(df_input)} entities")
else:
    file_path = os.path.join(os.getcwd(), INPUT_FOLDER, INPUT_FILE)
    df_input = entity_mgr.load_input_file(file_path)

print(f"\n📊 Loaded {len(df_input)} entities")
print(f"📋 Columns: {list(df_input.columns)}")
print(f"\n🌾 Unique Farms: {df_input['Farm'].nunique()}")
print(f"🌽 Unique Crops: {df_input['Crop'].unique().tolist()}")

display(df_input.head(10))

## ➕ Step 4: Create Entities

Creates the full hierarchy: `Farm → Field → Seasonfield`

**Note:** If the same Farm name appears multiple times, it will be created once and reused.

In [ ]:
# Create entities from input DataFrame
results_creation = entity_mgr.create_entities_from_dataframe(df_input, verbose=True)

# Display results
display(results_creation[['Farm', 'Seasonfield', 'Crop', 'Farm_Id', 'Field_Id', 'Seasonfield_Id', 'Status', 'Details']].head(20))

# Export results
entity_mgr.export_results(results_creation, "creation", manager.output_result_dir)

## 📤 Step 5: Export / Modification

- **Export**: Retrieve existing entities and export to CSV
- **Modify**: Update existing entities from CSV

In [ ]:
# ============ CONFIGURATION ============
OPERATION = "export"                    # "export" or "modify"
EXPORT_TYPE = "seasonfields"            # "farms", "fields", "seasonfields"
MODIFY_FILE = "entities_modify.csv"     # CSV for modifications

# Filters for export
EXPORT_GROWER_ID = None                 # Filter by grower ID
EXPORT_FARM_ID = None                   # Filter by farm ID
EXPORT_FIELD_ID = None                  # Filter by field ID
# ========================================

if OPERATION == "export":
    print(f"📤 Exporting {EXPORT_TYPE}...")
    
    if EXPORT_TYPE == "farms":
        df_export = entity_mgr.get_farms(grower_id=EXPORT_GROWER_ID)
    elif EXPORT_TYPE == "fields":
        df_export = entity_mgr.get_fields(farm_id=EXPORT_FARM_ID)
    elif EXPORT_TYPE == "seasonfields":
        df_export = entity_mgr.get_seasonfields(field_id=EXPORT_FIELD_ID)
    
    print(f"📊 Retrieved {len(df_export)} {EXPORT_TYPE}")
    display(df_export.head(20))
    
    if not df_export.empty:
        entity_mgr.export_results(df_export, "export", manager.output_result_dir)
        print("\n💡 Tip: Use exported CSV in Step 6 for deletion")

elif OPERATION == "modify":
    print(f"✏️ Loading modification file...")
    modify_path = os.path.join(os.getcwd(), INPUT_FOLDER, MODIFY_FILE)
    df_modify = entity_mgr.load_input_file(modify_path)
    
    print(f"\n⚠️ Modification requires manual implementation based on your needs.")
    print("Available update methods:")
    print("  - entity_mgr.update_farm(farm_id, farm_name=..., grower_id=...)")
    print("  - entity_mgr.update_field(field_id, field_name=..., geometry=..., farm_id=...)")
    print("  - entity_mgr.update_seasonfield(sf_id, geometry=..., sowing_date=..., crop_code=..., name=...)")
    
    display(df_modify.head())

## 🗑️ Step 6: Delete Entities

Delete entities using a CSV file with IDs.

**Required columns:**
- `Seasonfield_Id` → Seasonfield to delete
- `Field_Id` → Field to delete
- `Farm_Id` (optional) → Farm to delete (⚠️ deletes all fields under it)

**Tip:** Use the export from Step 4 (creation results) or Step 5 (export) as input.

In [ ]:
# ============ CONFIGURATION ============
USE_CREATED_IDS = True                                  # True = reuse IDs from Step 4 results, False = load CSV
DELETE_FILE = "entities_creation_20251217_120000.csv"  # CSV with IDs to delete (when USE_CREATED_IDS=False)
DELETE_FARMS = False                                    # ⚠️ Also delete farms?
CONFIRM_DELETE = False                                  # ⚠️ Set True to enable deletion
# ========================================

if USE_CREATED_IDS:
    # Reuse the entities just created in Step 4 — pulls Seasonfield_Id / Field_Id / Farm_Id
    # straight from results_creation, so no external file is needed.
    if 'results_creation' not in dir():
        raise RuntimeError("results_creation not found — run Step 4 first, or set USE_CREATED_IDS=False.")
    df_delete = results_creation[results_creation['Status'] == 'CREATED'].copy()
    print(f"♻️ Using {len(df_delete)} IDs from Step 4 (results_creation, Status==CREATED)")
else:
    delete_path = os.path.join(os.getcwd(), INPUT_FOLDER, DELETE_FILE)
    df_delete = entity_mgr.load_input_file(delete_path)

# Check required columns
id_columns = ['Seasonfield_Id', 'Field_Id', 'Farm_Id']
available_ids = [col for col in id_columns if col in df_delete.columns]
print(f"📋 ID columns found: {available_ids}")

print(f"\n⚠️ {len(df_delete)} entities to delete:")
display(df_delete[available_ids].head(10))

if CONFIRM_DELETE:
    results_deletion = entity_mgr.delete_entities_from_dataframe(
        df_delete, 
        delete_farms=DELETE_FARMS, 
        verbose=True
    )
    display(results_deletion[['Seasonfield', 'Status', 'Details']].head(20))
    entity_mgr.export_results(results_deletion, "deletion", manager.output_result_dir)
else:
    print("\n❌ Deletion NOT confirmed. Set CONFIRM_DELETE = True to proceed.")